# Tiền xử lý dữ liệu cho PhoBERT

Chọn kernel Python của môi trường Conda `myenv` và chạy các cell từ thư mục dự án. Notebook đọc `data_raw_all.csv`, chuẩn hóa và tách từ cột `sentences` bằng VnCoreNLP, rồi lưu `data_preprocess_all.csv`. Dữ liệu đầu vào đã được gộp và quy đổi nhãn.

In [ ]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import pandas as pd
from preprocess_text import preprocess_text

input_path = Path("data_raw_all.csv")
output_path = Path("data_preprocess_all.csv")
columns = ["sentences", "label"]

data = pd.read_csv(input_path, encoding="utf-8-sig", keep_default_na=False)
if list(data.columns) != columns:
    raise ValueError("CSV đầu vào phải có hai cột sentences và label.")
if data.empty or not data["label"].isin([0, 1, 2]).all():
    raise ValueError("CSV phải có dữ liệu và nhãn thuộc {0, 1, 2}.")

print(f"Số mẫu đầu vào: {len(data):,}")

## Tách từ bằng VnCoreNLP

Hàm `preprocess_text` chuẩn hóa Unicode và khoảng trắng, sau đó tách từ. Mô hình được tải một lần cho mỗi kernel.

In [ ]:
segmented = []
for index, sentence in enumerate(data["sentences"], start=1):
    text = preprocess_text(sentence)
    if not text.strip():
        raise ValueError(f"Văn bản rỗng sau khi tách từ tại mẫu {index}.")
    segmented.append(text)
    if index % 10000 == 0:
        print(f"Đã xử lý {index:,}/{len(data):,} mẫu", flush=True)

result = data.assign(sentences=segmented)

## Kiểm tra và lưu CSV

In [ ]:
with NamedTemporaryFile(
    dir=output_path.parent, prefix=output_path.stem + "_", suffix=".tmp", delete=False
) as temporary:
    temporary_path = Path(temporary.name)

try:
    result.to_csv(temporary_path, index=False, encoding="utf-8-sig")
    saved = pd.read_csv(temporary_path, encoding="utf-8-sig", keep_default_na=False)
    pd.testing.assert_frame_equal(result, saved)
    pd.testing.assert_series_equal(data["label"], saved["label"])
    temporary_path.replace(output_path)
finally:
    temporary_path.unlink(missing_ok=True)

print(f"Đã lưu {len(saved):,} mẫu vào {output_path}")
print(saved["label"].value_counts().sort_index().to_string())